# ECC skalyar ko‘paytirishni optimallashtirish — eksperimental notebook

Binary, NAF, wNAF va fixed-window algoritmlari; kichik F17 va katta secp256k1 maydonlari; 50 takroriy o‘lchov; to‘g‘rilik testlari; xotira modeli; statistik tahlil; grafiklar; CSV va Excel eksporti.

**Metodologiya:** vaqtlar notebook ishga tushirilganda real o‘lchanadi. Jadval xotirasi `estimated` — koordinata payloadi bo‘yicha nazariy baho, Python jarayonining umumiy RAM/RSS qiymati emas. Affine koordinatalar ishlatiladi; bu tushunarli bazaviy implementatsiya, yon-kanal hujumlariga chidamli production kriptografik kutubxona emas.


In [ ]:
# 1. Kutubxonalar, egri chiziqlar va affine nuqta amallari
import math, random, statistics, time, platform, sys
from dataclasses import dataclass
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

REPEATS = 50
SEED = 20261022
OUT = Path("ecc_scalar_results")
OUT.mkdir(exist_ok=True)

@dataclass(frozen=True)
class Curve:
    name: str
    p: int
    a: int
    b: int
    G: tuple

O = None
SMALL = Curve("Small_F17", 17, 2, 2, (5, 1))
SECP256K1 = Curve(
    "secp256k1",
    0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFC2F,
    0, 7,
    (0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798,
     0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8)
)

def on_curve(P, c):
    return P is O or (P[1]*P[1] - (P[0]**3+c.a*P[0]+c.b)) % c.p == 0

def neg(P, c):
    return O if P is O else (P[0], (-P[1]) % c.p)

def add(P, Q, c):
    if P is O: return Q
    if Q is O: return P
    x1,y1=P; x2,y2=Q; p=c.p
    if x1 == x2 and (y1+y2) % p == 0: return O
    if P == Q:
        if y1 % p == 0: return O
        num=(3*x1*x1+c.a) % p; den=(2*y1) % p
    else:
        num=(y2-y1) % p; den=(x2-x1) % p
    lam=num*pow(den, -1, p) % p
    x3=(lam*lam-x1-x2) % p
    return (x3, (lam*(x1-x3)-y1) % p)

def double(P, c):
    return add(P, P, c)

print("Python:", sys.version.split()[0], "| Platforma:", platform.platform())
print("Takrorlashlar:", REPEATS, "| Natijalar papkasi:", OUT.resolve())
print("Generatorlar egri chiziqda:", on_curve(SMALL.G, SMALL), on_curve(SECP256K1.G, SECP256K1))

# 3. Skalyar kodlash va algoritmlar — tuzatilgan hisoblagichlar
def binary(k, P, c):
    if k < 0:
        return binary(-k, neg(P, c), c)
    if k == 0 or P is O:
        return O, 0, 0, 0
    R = P
    D = A = 0
    for bit in bin(k)[3:]:
        R = double(R, c); D += 1
        if bit == "1":
            R = add(R, P, c); A += 1
    return R, D, A, 0

def naf_digits(k):
    ds = []
    n = k
    while n > 0:
        if n & 1:
            d = 2 - (n % 4)
            ds.append(d)
            n -= d
        else:
            ds.append(0)
        n //= 2
    return ds

def wnaf_digits(k, w):
    if w < 2:
        raise ValueError("wNAF uchun w >= 2")
    ds = []
    n = k
    radix, half = 1 << w, 1 << (w - 1)
    while n:
        if n & 1:
            d = n % radix
            if d >= half:
                d -= radix
            ds.append(d)
            n -= d
        else:
            ds.append(0)
        n //= 2
    return ds

def signed_mul(ds, P, c, table=None):
    # Birinchi noldan farqli raqam R ga bevosita yoziladi.
    # O + Q guruh qo‘shishi hisoblanmaydi.
    R = O
    D = A = 0
    for d in reversed(ds):
        if R is not O:
            R = double(R, c)
            D += 1
        if d:
            if abs(d) == 1:
                Q = P
            else:
                if table is None:
                    raise ValueError("Katta NAF raqami uchun jadval kerak")
                Q = table[abs(d)]
            if d < 0:
                Q = neg(Q, c)
            if R is O:
                R = Q
            else:
                R = add(R, Q, c)
                A += 1
    return R, D, A

def naf(k, P, c):
    if k < 0:
        return naf(-k, neg(P, c), c)
    if k == 0:
        return O, 0, 0, 0
    R, D, A = signed_mul(naf_digits(k), P, c)
    return R, D, A, 0

def odd_table(P, c, w):
    # P, 3P, 5P, ..., (2^(w-1)-1)P
    table = {1: P}
    twoP = double(P, c)
    cur = P
    pre_adds = 0
    for odd in range(3, 1 << (w - 1), 2):
        cur = add(cur, twoP, c)
        table[odd] = cur
        pre_adds += 1
    return table, pre_adds

def wnaf(k, P, c, w):
    if k < 0:
        return wnaf(-k, neg(P, c), w)
    if k == 0:
        return O, 0, 0, 0
    table, pre_adds = odd_table(P, c, w)
    R, D, A = signed_mul(wnaf_digits(k, w), P, c, table)
    return R, D, A, pre_adds

def fixed_window(k, P, c, w):
    if w < 1:
        raise ValueError("Fixed-window uchun w >= 1")
    if k < 0:
        return fixed_window(-k, neg(P, c), w)
    if k == 0:
        return O, 0, 0, 0
    radix = 1 << w
    table = {0: O, 1: P}
    pre_adds = 0
    for j in range(2, radix):
        table[j] = add(table[j - 1], P, c)
        pre_adds += 1
    ds, n = [], k
    while n:
        ds.append(n % radix)
        n //= radix
    R = O
    D = A = 0
    for d in reversed(ds):
        if R is not O:
            for _ in range(w):
                R = double(R, c)
                D += 1
        if d:
            if R is O:
                R = table[d]
            else:
                R = add(R, table[d], c)
                A += 1
    return R, D, A, pre_adds

ALG = {"Binary": binary, "NAF": naf, "wNAF": wnaf, "FixedWindow": fixed_window}
print("Tuzatilgan algoritmlar tayyor. Boshlang‘ich O nuqtaga qo‘shish amallari hisoblagichga kiritilmaydi.")


## 1. Matematik to‘g‘rilik testlari

Har bir usul natijasi Binary algoritmi natijasi bilan solishtiriladi; natija egri chiziq tenglamasini ham qanoatlantirishi kerak. Kichik maydon — mantiqiy test uchun; secp256k1 — katta maydon sinovi uchun.


In [ ]:
def correctness_tests():
    rows=[]
    for c, ks in [(SMALL,[1,2,3,13,25,100]), (SECP256K1,[1,2,3,13,12345,(1<<127)+123])]:
        for k in ks:
            ref,*_=binary(k,c.G,c)
            for name in ["Binary","NAF"]:
                r,D,A,pre=ALG[name](k,c.G,c,3) if name in ("wNAF","FixedWindow") else ALG[name](k,c.G,c)
                rows.append({"curve":c.name,"k":k,"algorithm":name,"matches_binary":r==ref,"on_curve":on_curve(r,c),"D":D,"A":A,"precompute_adds":pre})
            for w in [2,3,4,5]:
                for name in ["wNAF","FixedWindow"]:
                    r,D,A,pre=ALG[name](k,c.G,c,w)
                    rows.append({"curve":c.name,"k":k,"algorithm":f"{name}_w{w}","matches_binary":r==ref,"on_curve":on_curve(r,c),"D":D,"A":A,"precompute_adds":pre})
    df=pd.DataFrame(rows)
    assert df.matches_binary.all(), "Algoritmlar chiqishi mos kelmadi"
    assert df.on_curve.all(), "Natija egri chiziqda emas"
    return df

correctness_df=correctness_tests()
display(correctness_df.head(15))
print("To‘g‘rilik testlari muvaffaqiyatli.")


## 2. Benchmark — 50 ta takror

Katta maydon uchun 64/128/192/256 bitli skalyarlar, kichik maydon uchun 8/16/32/64 bitli skalyarlar olinadi. Har bir konfiguratsiyada 50 ta tasodifiy skalyar yaratiladi; vaqtlar `perf_counter()` bilan o‘lchanadi. Natijalar aynan shu kompyuterda olingan o‘lchovlardir.


In [ ]:
# 5. Benchmark — 50 takror, individual samples va alohida precompute/main timing
def scalar(bits, rng):
    return rng.randrange(1 << (bits - 1), 1 << bits)

def table_payload(c, alg, w):
    coord_bytes = (c.p.bit_length() + 7) // 8
    n = (1 << (w - 2)) if alg == "wNAF" else ((1 << w) - 1 if alg == "FixedWindow" else 0)
    return n * 2 * coord_bytes

def timed_wnaf(k, P, c, w):
    if k == 0:
        return O, 0, 0, 0, 0.0, 0.0
    t0 = time.perf_counter()
    table, pre_adds = odd_table(P, c, w)
    pre_time = time.perf_counter() - t0
    t1 = time.perf_counter()
    R, D, A = signed_mul(wnaf_digits(k, w), P, c, table)
    main_time = time.perf_counter() - t1
    return R, D, A, pre_adds, pre_time, main_time

def timed_fixed(k, P, c, w):
    if k == 0:
        return O, 0, 0, 0, 0.0, 0.0
    t0 = time.perf_counter()
    radix = 1 << w
    table = {0: O, 1: P}
    pre_adds = 0
    for j in range(2, radix):
        table[j] = add(table[j - 1], P, c)
        pre_adds += 1
    pre_time = time.perf_counter() - t0
    ds, n = [], k
    while n:
        ds.append(n % radix)
        n //= radix
    t1 = time.perf_counter()
    R = O
    D = A = 0
    for d in reversed(ds):
        if R is not O:
            for _ in range(w):
                R = double(R, c); D += 1
        if d:
            if R is O:
                R = table[d]
            else:
                R = add(R, table[d], c); A += 1
    main_time = time.perf_counter() - t1
    return R, D, A, pre_adds, pre_time, main_time

def run_timed(alg, k, c, w):
    if alg == "wNAF":
        return timed_wnaf(k, c.G, c, w)
    if alg == "FixedWindow":
        return timed_fixed(k, c.G, c, w)
    t0 = time.perf_counter()
    result, D, A, pre = ALG[alg](k, c.G, c)
    elapsed = time.perf_counter() - t0
    return result, D, A, pre, 0.0, elapsed

def bench_one(c, bits, alg, w, repeats=REPEATS):
    rng = random.Random(SEED + bits + w + len(alg) + c.p.bit_length())
    samples = []
    correctness = True
    for rep in range(repeats):
        k = scalar(bits, rng)
        reference, *_ = binary(k, c.G, c)
        result, D, A, pre_adds, pre_time, main_time = run_timed(alg, k, c, w)
        total = pre_time + main_time
        correctness = correctness and (result == reference) and on_curve(result, c)
        samples.append({
            "curve": c.name, "field_bits": c.p.bit_length(),
            "scalar_bits": bits, "algorithm": alg,
            "window_w": w if alg in ("wNAF", "FixedWindow") else None,
            "repeat_index": rep + 1, "scalar_k": k,
            "precompute_time_s_measured": pre_time,
            "main_time_s_measured": main_time,
            "total_time_s_measured": total,
            "doublings_counted": D, "additions_counted": A,
            "precompute_additions_counted": pre_adds,
            "table_payload_bytes_estimated": table_payload(c, alg, w),
            "correctness": result == reference and on_curve(result, c),
            "time_status": "measured with perf_counter on this runtime",
            "memory_status": "estimated coordinate payload; not process RSS"
        })
    df = pd.DataFrame(samples)
    # Repeat-level measurements are preserved in the raw table.
    return df

RUN_FULL_EXPERIMENT = True
raw_parts = []
if RUN_FULL_EXPERIMENT:
    configs = [(SMALL, [8, 16, 32, 64]), (SECP256K1, [64, 128, 192, 256])]
    total_configs = sum(len(bits_list) * (2 + 5 + 4) for _, bits_list in configs)
    print(f"Konfiguratsiyalar: {total_configs}; har birida {REPEATS} takror.")
    for c, bit_list in configs:
        for bits in bit_list:
            for alg in ["Binary", "NAF"]:
                part = bench_one(c, bits, alg, 3)
                raw_parts.append(part)
                print("Tugadi:", c.name, bits, alg)
            for w in [2, 3, 4, 5, 6]:
                part = bench_one(c, bits, "wNAF", w)
                raw_parts.append(part)
                print("Tugadi:", c.name, bits, "wNAF", w)
            for w in [2, 3, 4, 5]:
                part = bench_one(c, bits, "FixedWindow", w)
                raw_parts.append(part)
                print("Tugadi:", c.name, bits, "FixedWindow", w)
    benchmark_raw_df = pd.concat(raw_parts, ignore_index=True)
    display(benchmark_raw_df.head())
    print("Raw jadval har bir takrorning alohida o‘lchovini saqlaydi.")
else:
    benchmark_raw_df = pd.DataFrame()


## 3. Xotira cheklovi va parametr tanlash

Quyidagi xotira — jadval koordinatalarining payload bahosi, Python obyekt overhead'i emas. Agar benchmark bajarilgan bo‘lsa, xotira budjetiga mos variantlar ichida o‘lchangan o‘rtacha vaqti eng kichik variant tanlanadi. Agar o‘lchovlar bo‘lmasa, vaqt bo‘yicha g‘olib e’lon qilinmaydi.


In [ ]:
MEMORY_BUDGET_BYTES=16*1024

def optimize_by_measured_time(df,c,alg,widths,budget):
    candidates=[]
    for w in widths:
        mem=table_payload(c,alg,w)
        candidates.append({"curve":c.name,"algorithm":alg,"w":w,
                           "table_points":(1<<(w-2)) if alg=="wNAF" else (1<<w)-1,
                           "table_payload_bytes_estimated":mem,"budget_bytes":budget,
                           "feasible_by_payload_estimate":mem<=budget})
    tab=pd.DataFrame(candidates)
    feasible=tab[tab.feasible_by_payload_estimate]
    measured=None
    if not feasible.empty and not df.empty:
        sub=df[(df.curve==c.name)&(df.algorithm==alg)&(df.window_w.isin(feasible.w))]
        if not sub.empty:
            means=sub.groupby("window_w")["total_time_s_measured"].mean()
            measured=int(means.idxmin())
    return tab, measured

mem_tabs=[]; choices=[]
for c in [SMALL,SECP256K1]:
    for alg,widths in [("wNAF",[2,3,4,5,6]),("FixedWindow",[2,3,4,5])]:
        tab,best=optimize_by_measured_time(benchmark_raw_df,c,alg,widths,MEMORY_BUDGET_BYTES)
        mem_tabs.append(tab)
        choices.append({"curve":c.name,"algorithm":alg,"budget_bytes":MEMORY_BUDGET_BYTES,
                        "best_w_by_measured_time":best,
                        "note":"None bo‘lsa, mavjud benchmark ichida tanlov yo‘q."})
memory_df=pd.concat(mem_tabs,ignore_index=True)
memory_choice_df=pd.DataFrame(choices)
display(memory_df)
display(memory_choice_df)


## 4. Statistik tahlil va grafiklar

`mean_time_s_measured`, median, standart og‘ish, minimum va maksimum o‘lchangan vaqtlar asosida hisoblanadi. Grafiklar har bir egri chiziq uchun alohida chiqariladi.


In [ ]:
# 8. Statistik tahlil va grafiklar
def summarize(df):
    keys = ["curve", "scalar_bits", "algorithm", "window_w"]
    return (df.groupby(keys, dropna=False)
            .agg(
                n=("total_time_s_measured", "count"),
                mean_total_time_s=("total_time_s_measured", "mean"),
                median_total_time_s=("total_time_s_measured", "median"),
                sd_total_time_s=("total_time_s_measured", "std"),
                min_total_time_s=("total_time_s_measured", "min"),
                max_total_time_s=("total_time_s_measured", "max"),
                mean_precompute_time_s=("precompute_time_s_measured", "mean"),
                mean_main_time_s=("main_time_s_measured", "mean"),
                mean_doublings=("doublings_counted", "mean"),
                mean_additions=("additions_counted", "mean"),
                mean_precompute_additions=("precompute_additions_counted", "mean"),
                all_correct=("correctness", "all")
            ).reset_index())

if not benchmark_raw_df.empty:
    summary_df = summarize(benchmark_raw_df)
    display(summary_df.head(20))
    for cname in benchmark_raw_df["curve"].unique():
        sub = benchmark_raw_df[benchmark_raw_df["curve"] == cname]
        plt.figure(figsize=(10, 5))
        for alg in sub["algorithm"].unique():
            s = sub[sub["algorithm"] == alg].groupby("scalar_bits")["total_time_s_measured"].mean()
            plt.plot(s.index, s.values, marker="o", label=alg)
        plt.xlabel("Skalyar uzunligi (bit)")
        plt.ylabel("O‘rtacha umumiy vaqt (soniya)")
        plt.title(f"{cname}: skalyar uzunligi va vaqt")
        plt.grid(True); plt.legend(); plt.tight_layout(); plt.show()

    wdf = benchmark_raw_df[benchmark_raw_df["algorithm"] == "wNAF"]
    if not wdf.empty:
        plt.figure(figsize=(9, 5))
        for cname in wdf["curve"].unique():
            sub = wdf[wdf["curve"] == cname]
            s = sub.groupby("window_w")["total_time_s_measured"].mean()
            plt.plot(s.index, s.values, marker="o", label=cname)
        plt.xlabel("wNAF oyna kengligi w")
        plt.ylabel("O‘rtacha umumiy vaqt (soniya)")
        plt.title("Oyna kengligining umumiy vaqtga ta’siri")
        plt.grid(True); plt.legend(); plt.tight_layout(); plt.show()
else:
    print("Avval benchmarkni ishga tushiring.")


## 5. CSV va Excel eksporti

Eksportlar `ecc_scalar_results` papkasiga yoziladi. Excel faylida to‘g‘rilik, xom benchmark, umumlashtirilgan natijalar va xotira modeli alohida varaqlarda saqlanadi.


In [ ]:
# 9. CSV va Excel eksporti
correctness_path = OUT / "correctness_tests.csv"
correctness_df.to_csv(correctness_path, index=False, encoding="utf-8-sig")
exported = {"correctness_csv": str(correctness_path.resolve())}

if not benchmark_raw_df.empty:
    raw_path = OUT / "benchmark_raw.csv"
    summary_path = OUT / "benchmark_summary.csv"
    benchmark_raw_df.to_csv(raw_path, index=False, encoding="utf-8-sig")
    summary_df = summarize(benchmark_raw_df)
    summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")
    xlsx_path = OUT / "ecc_scalar_multiplication_results.xlsx"
    with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
        correctness_df.to_excel(writer, sheet_name="Correctness", index=False)
        benchmark_raw_df.to_excel(writer, sheet_name="Benchmark_Raw", index=False)
        summary_df.to_excel(writer, sheet_name="Summary", index=False)
        memory_df.to_excel(writer, sheet_name="Memory_Model", index=False)
        memory_choice_df.to_excel(writer, sheet_name="Memory_Choice", index=False)
    exported.update({"benchmark_csv": str(raw_path.resolve()),
                     "summary_csv": str(summary_path.resolve()),
                     "excel": str(xlsx_path.resolve())})
exported


## Dissertatsiya uchun ehtiyot choralari

- Benchmark vaqtlarini faqat notebook bajarilgan kompyuter va Python muhiti uchun real o‘lchov deb ko‘rsating.
- Jadval xotirasi `estimated`; haqiqiy process RAM o‘lchovi emas.
- Kichik F17 natijalari algoritm mantig‘ini tekshirishga xizmat qiladi, katta razryadli kriptografik samaradorlikni ifodalamaydi.
- secp256k1 generatorining nuqta tenglamasiga tegishliligi tekshiriladi, ammo notebook tub modul va generator tartibini formal sertifikatlash vositasi emas.
- Affine koordinatalardagi modular inversiyalar tezlikka sezilarli ta’sir qiladi. Barcha algoritmlar ayni amallar realizatsiyasidan foydalanadi, lekin natijalarni boshqa kutubxona yoki proyektiv koordinatalar bilan bevosita tenglashtirmang.
- Maxfiy kalitlar uchun ushbu kodni production kriptografiyada ishlatmang: u constant-time himoyani kafolatlamaydi.
